# Inversion of borehole temperatures

We estimate the geothermal gradient and surface temperature from four borehole
temperature measurements by linear least squares. Every quantity carries its
units in its type, so fortl checks that the inversion is dimensionally consistent.

Each cell sees the definitions from earlier cells; the output shows the type and
value of the cell's last definition. Try **Shift-Tab** on a name to see its type,
and **Tab** to complete names.

## Data

Thermistor depths:

In [ ]:
z1 : Float[Unit[m]] = 100.0
z2 : Float[Unit[m]] = 200.0
z3 : Float[Unit[m]] = 300.0
z4 : Float[Unit[m]] = 500.0

z4 : Float[Unit[m]]
  = 500.0

and measured temperatures:

In [ ]:
t1 : Float[Unit[K]] = 285.7
t2 : Float[Unit[K]] = 288.1
t3 : Float[Unit[K]] = 290.6
t4 : Float[Unit[K]] = 293.2

t4 : Float[Unit[K]]
  = 293.2

<img src="../articles/figures/borehole-fit.svg" alt="Borehole data and least-squares fit" width="420">

## Normal equations

For the model $T(z) = T_0 + G\,z$ the normal equations are
$$
\begin{pmatrix} n & \Sigma z \\ \Sigma z & \Sigma z^2 \end{pmatrix}
\begin{pmatrix} T_0 \\ G \end{pmatrix}
=
\begin{pmatrix} \Sigma T \\ \Sigma zT \end{pmatrix},
$$
whose entries all have different units:

In [ ]:
n : Float[Unit[1]] = 4.0

sum_z   : Float[Unit[m]]     = z1 + z2 + z3 + z4
sum_zz  : Float[Unit[m^2]]   = z1*z1 + z2*z2 + z3*z3 + z4*z4
sum_t   : Float[Unit[K]]     = t1 + t2 + t3 + t4
sum_zt  : Float[Unit[m * K]] = z1*t1 + z2*t2 + z3*t3 + z4*t4

sum_zt : Float[Unit[(m * K)]]
      = 319970.0

## Cramer's rule

In [ ]:
det : Float[Unit[m^2]] = n * sum_zz - sum_z * sum_z

det : Float[Unit[m^2]]
   = 350000.0

In [ ]:
grad : Float[Unit[K / m]] = (n * sum_zt - sum_z * sum_t) / det

grad : Float[Unit[(K * m^-1)]]
    = 2.4999375e-2

In [ ]:
t0 : Float[Unit[K]] = (sum_zz * sum_t - sum_z * sum_zt) / det

t0 : Float[Unit[K]]
  = 283.15024

A cell that is just an expression is evaluated against the definitions so far (and not kept). Here, the model's prediction at the deepest thermistor:

In [ ]:
t0 + grad * z4

Float[Unit[K]]
= 293.15

A unit mistake is caught before anything runs:

In [ ]:
bad_grad : Float[Unit[K / m]] = (n * sum_zz - sum_z * sum_t) / det

TypeError: 

## Heat flow

Fourier's law, $q = k\,G$:


In [ ]:
conductivity : Float[Unit[J / (s * m * K)]] = 3.0
heat_flow : Float[Unit[J / (s * m^2)]] = conductivity * grad

heat_flow : Float[Unit[(J * (s * m^2)^-1)]]
         = 7.4998125e-2

## Working with files

`%file` shows part of a file from the repository (click the name to open it in
the editor) and runs the whole file:

In [ ]:
%file ../examples/geotherm-inversion.frtl 50-63

"""Forward model, used to check the fit"""
predict : Float[Unit[m]] -> Float[Unit[K]] = (lambda z: t0 + grad * z)

"""Data residual at a single observation"""
residual : Float[Unit[m]] -> Float[Unit[K]] -> Float[Unit[K]] = (
    lambda z: lambda t: t - predict(z))

r1 : Float[Unit[K]] = residual(z1)(t1)
r2 : Float[Unit[K]] = residual(z2)(t2)
r3 : Float[Unit[K]] = residual(z3)(t3)
r4 : Float[Unit[K]] = residual(z4)(t4)

"""Root-mean-square misfit between data and fitted model"""
rms_misfit : Float[Unit[K]] = sqrt((r1*r1 + r2*r2 + r3*r3 + r4*r4) / n)

((Float[Unit[K]] * Float[Unit[(K * m^-1)]]) * Float[Unit[K]]) * Float[Unit[(J * (s * m^2)^-1)]]
= (((283.15024, 2.4999375e-2), 5.0003156e-2), 7.4998125e-2)

`show=` picks out a particular binding:

In [ ]:
%file ../examples/geotherm-inversion.frtl 62-63 show=rms_misfit

"""Root-mean-square misfit between data and fitted model"""
rms_misfit : Float[Unit[K]] = sqrt((r1*r1 + r2*r2 + r3*r3 + r4*r4) / n)

rms_misfit : Float[Unit[K]]
          = 5.0003156e-2

`%load` brings a file's definitions into the session, like `:l` in `fortli`:

In [ ]:
%reset

Session cleared.


In [ ]:
%load ../examples/geotherm-inversion.frtl

Loaded 26 definitions from ../examples/geotherm-inversion.frtl: z1, z2, z3, z4, t1, t2, t3, t4, n, sum_z, sum_zz, sum_t, sum_zt, det, grad, t0, predict, residual, r1, r2, r3, r4, rms_misfit, conductivity, heat_flow, it


In [ ]:
heat_flow / grad

Float[Unit[(K^-1 * (m^-1 * (s^-1 * J)))]]
= 3.0